구글 드라이브 웹 화면에는 `train/images` 및 `labels` 폴더가 정상적으로 존재하지만, 코랩(Colab) 파이썬 코드에서 `0장`으로 감지된 원인은 **구글 드라이브 가상 마운트(FUSE)의 파일 동기화 지연** 또는 **마운트 캐시 블록** 때문입니다.

웹 UI에 올린 파일이 코랩의 가상 파일 시스템(`/content/drive/MyDrive`)에 즉시 반영되지 않으면, 폴더 구조만 보이고 내부 이미지 파일 리스트(`os.listdir`)를 파이썬이 읽지 못하는 현상이 발생합니다.

---

### 원인 해결 및 동기화 강제 적용 통합 스크립트

구글 드라이브 마운트 캐시를 강제로 새로고침(`force_remount=True`)하여 파일 동기화를 완료한 뒤, 증강 및 YOLOv8 학습을 진행하는 수정 코드입니다. 코랩 셀에 복사하여 실행하세요.

In [ ]:
import os
import glob
import shutil
import random
import cv2
import torch
import numpy as np
import albumentations as A
from google.colab import drive
from ultralytics import YOLO

# 1. 구글 드라이브 강제 재마운트 (동기화 지연 및 캐시 문제 해결)
print("🔄 구글 드라이브 동기화 및 강제 재마운트 진행 중...")
drive.mount('/content/drive', force_remount=True)

DRIVE_DATA_DIR = "/content/drive/MyDrive/Pothole"
LOCAL_DATA_DIR = "/content/Pothole_Local"

# 2. 구글 드라이브 내 train/images 인식 직접 검증
drive_train_img_dir = os.path.join(DRIVE_DATA_DIR, "train/images")

if os.path.exists(drive_train_img_dir):
    drive_imgs = [f for f in os.listdir(drive_train_img_dir) 
                  if os.path.splitext(f)[1].lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.webp']]
    print(f"🔎 [드라이브 직접 확인] train/images 내 이미지 파일: {len(drive_imgs)}장 감지됨")
else:
    raise FileNotFoundError(f"❌ 구글 드라이브 경로를 찾을 수 없습니다: {drive_train_img_dir}")

if len(drive_imgs) == 0:
    print("⚠️ 구글 드라이브 'Pothole/train/images' 폴더 안에 실제 .jpg / .png 파일이 들어있는지 웹 드라이브에서 재확인해 주세요.")

# 3. 로컬 작업 디렉토리(Pothole_Local)로 고속 복사
print("📦 [Pothole -> Pothole_Local] 로컬 고속 복사 중...")
if os.path.exists(LOCAL_DATA_DIR):
    shutil.rmtree(LOCAL_DATA_DIR)

# .cache 파일 및 불필요한 가중치 파일 제외 후 복사
shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR, ignore=shutil.ignore_patterns('runs*', '*.pt', '*.yaml', '*.cache'))
print("✅ 로컬 복사 완료!")

# 4. 도로 환경 특화 오프라인 데이터 증강 모듈
class PotholeDataAugmenter:
    def __init__(self):
        self.base_transform = A.Compose([
            A.MotionBlur(blur_limit=(7, 15), p=0.5),
            A.Defocus(radius=(3, 6), p=0.3),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=0.3)
        ])

    def apply_wet_asphalt_and_glint(self, image):
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        lab = cv2.merge((l, a, b))
        wet_img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

        h, w, _ = image.shape
        if random.random() < 0.6:
            glint_mask = np.zeros((h, w), dtype=np.float32)
            cx, cy = random.randint(int(w*0.2), int(w*0.8)), random.randint(int(h*0.3), int(h*0.9))
            rx, ry = random.randint(30, 100), random.randint(10, 40)
            angle = random.randint(0, 180)
            cv2.ellipse(glint_mask, (cx, cy), (rx, ry), angle, 0, 360, 1.0, -1)
            glint_mask = cv2.GaussianBlur(glint_mask, (77, 77), 0)
            glint_bgr = cv2.cvtColor((glint_mask * 180).astype(np.uint8), cv2.COLOR_GRAY2BGR)
            wet_img = cv2.addWeighted(wet_img, 1.0, glint_bgr, 0.4, 0)
        return wet_img

    def synthesize_shadows(self, image):
        h, w, _ = image.shape
        shadow_mask = np.ones((h, w), dtype=np.float32)
        num_polygons = random.randint(1, 3)
        for _ in range(num_polygons):
            pts = np.array([
                [random.randint(0, w), random.randint(0, int(h*0.5))],
                [random.randint(0, w), random.randint(0, int(h*0.5))],
                [random.randint(0, w), random.randint(int(h*0.5), h)],
                [random.randint(0, w), random.randint(int(h*0.5), h)]
            ], np.int32)
            poly_mask = np.zeros((h, w), dtype=np.uint8)
            cv2.fillPoly(poly_mask, [pts], 255)
            shadow_factor = random.uniform(0.3, 0.55)
            shadow_mask[poly_mask == 255] *= shadow_factor
        return (image.astype(np.float32) * shadow_mask[:, :, np.newaxis]).astype(np.uint8)

    def transform_pipeline(self, image):
        img = self.apply_wet_asphalt_and_glint(image)
        img = self.synthesize_shadows(img)
        return self.base_transform(image=img)['image']

# 5. 오프라인 데이터 증강 진행
local_train_img_dir = os.path.join(LOCAL_DATA_DIR, "train/images")
AUG_TRAIN_IMG_DIR = os.path.join(LOCAL_DATA_DIR, "train/images_aug")
AUG_TRAIN_LBL_DIR = os.path.join(LOCAL_DATA_DIR, "train/labels_aug")

os.makedirs(AUG_TRAIN_IMG_DIR, exist_ok=True)
os.makedirs(AUG_TRAIN_LBL_DIR, exist_ok=True)

train_img_paths = [os.path.join(local_train_img_dir, f) for f in os.listdir(local_train_img_dir) 
                   if os.path.splitext(f)[1].lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.webp']]

print(f"🎨 [오프라인 데이터 증강] 대상 이미지 수: {len(train_img_paths)}장 준비 완료")

if len(train_img_paths) > 0:
    augmenter = PotholeDataAugmenter()
    for img_path in train_img_paths:
        filename = os.path.basename(img_path)
        base_name = os.path.splitext(filename)[0]
        lbl_path = os.path.join(LOCAL_DATA_DIR, "train/labels", base_name + ".txt")

        # 원본 이미지/라벨 복사
        shutil.copy2(img_path, os.path.join(AUG_TRAIN_IMG_DIR, filename))
        if os.path.exists(lbl_path):
            shutil.copy2(lbl_path, os.path.join(AUG_TRAIN_LBL_DIR, base_name + ".txt"))

        # 증강 2장 추가 생성
        img = cv2.imread(img_path)
        if img is None:
            continue
        for i in range(2):
            aug_img = augmenter.transform_pipeline(img)
            aug_filename = f"{base_name}_envaug_{i}.jpg"
            cv2.imwrite(os.path.join(AUG_TRAIN_IMG_DIR, aug_filename), aug_img)
            if os.path.exists(lbl_path):
                shutil.copy2(lbl_path, os.path.join(AUG_TRAIN_LBL_DIR, f"{base_name}_envaug_{i}.txt"))
    print("✅ [train/images_aug] 증강 데이터셋 생성 완료")

    # 6. data.yaml 자동 작성
    yaml_path = os.path.join(LOCAL_DATA_DIR, "data.yaml")
    yaml_content = f"""
path: {LOCAL_DATA_DIR}
train: train/images_aug
val: valid/images
test: test/images

nc: 1
names: ['pothole']
""".strip()

    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write(yaml_content)
    print("✅ [data.yaml] 작성 완료")

    # 7. YOLOv8 n/s/m 모델 순차 학습
    device_setting = 0 if torch.cuda.is_available() else "cpu"
    SAVE_PROJECT_DIR = os.path.join(DRIVE_DATA_DIR, "runs")

    models_to_train = [
        ("yolov8n", "yolov8n.pt", 16),
        ("yolov8s", "yolov8s.pt", 16),
        ("yolov8m", "yolov8m.pt", 8)
    ]

    for model_name, weights, batch_size in models_to_train:
        print(f"\n🚀 [{model_name}_pothole_env_aug] 학습 시작... (할당 장치: {device_setting})")
        model = YOLO(weights)
        model.train(
            data=yaml_path,
            epochs=150,
            imgsz=1024,
            batch=batch_size,
            optimizer="AdamW",
            lr0=0.001,
            patience=30,
            project=SAVE_PROJECT_DIR,
            name=f"{model_name}_pothole_env_aug",
            exist_ok=True,
            device=device_setting
        )
    print("\n🏆 [YOLOv8 n/s/m] 순차 학습 및 구글 드라이브 가중치 저장 완료!")